# 🏦 Agent Framework Skills — Financial Services

This notebook demonstrates **Agent Skills** with the Microsoft Agent Framework using Financial Services Industry (FSI) scenarios:

1. **File-based Skills** — Credit risk calculator with executable scripts
2. **Code-defined Skills** — Inline compliance checker with dynamic resources
3. **Combined Skills** — Agent with both file-based and code-defined skills

### What are Agent Skills?
Skills are portable packages of instructions, scripts, and resources that give agents specialized capabilities. They use **progressive disclosure** so agents only load context when needed:

1. **Advertise** (~100 tokens) — Names and descriptions in system prompt
2. **Load** — Full SKILL.md instructions on demand
3. **Read Resources** — Supplementary files as needed
4. **Run Scripts** — Execute bundled scripts with approval

## 1. Setup & Configuration

In [ ]:
import json
import os
from pathlib import Path
from textwrap import dedent

from dotenv import load_dotenv

from azure.identity.aio import AzureCliCredential
from agent_framework import (
    Skill, SkillScript, SkillFrontmatter, SkillsProvider, InlineSkill,
    AggregatingSkillsSource, DeduplicatingSkillsSource,
    FileSkillsSource, InMemorySkillsSource,
)
from agent_framework_foundry import FoundryChatClient

# Load environment variables from the repo-root .env (FOUNDRY_PROJECT_ENDPOINT, FOUNDRY_MODEL, etc.)
load_dotenv()

print("✅ Imports loaded")

## 2. File-Based Skills — Credit Risk Calculator

File-based skills are discovered from `SKILL.md` files in the filesystem. Our `fsi-skills/credit-risk-calculator/` folder contains:

```
credit-risk-calculator/
├── SKILL.md                    # Frontmatter + instructions
└── scripts/
    └── calculate_risk.py       # Executable risk scoring script
```

The `SkillsProvider` discovers these automatically and makes `load_skill`, `read_skill_resource`, and `run_skill_script` tools available to the agent.

In [ ]:
import subprocess
import sys

def fsi_script_runner(skill: Skill, script: SkillScript, args: dict | None = None) -> str:
    """Run a file-based skill script as a subprocess."""
    script_path = Path(skill.path) / script.path
    cmd = [sys.executable, str(script_path)]
    if args:
        for key, value in args.items():
            if value is not None:
                cmd.extend([f"--{key}", str(value)])
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    if result.returncode != 0:
        return f"Error: {result.stderr.strip()}"
    return result.stdout.strip()

# Create SkillsProvider pointing to our FSI skills directory
file_skills_provider = SkillsProvider.from_paths(
    Path(".").absolute() / "fsi-skills",
    script_runner=fsi_script_runner,
)

print("✅ File-based SkillsProvider configured")
print(f"   Skills directory: {Path('.').absolute() / 'fsi-skills'}")

In [ ]:
async def demo_file_based_skills():
    """Demonstrate file-based skills with credit risk assessment."""
    async with AzureCliCredential(process_timeout=30) as credential:
        client = FoundryChatClient(credential=credential)
        agent = client.as_agent(
            name="CreditRiskAgent",
            instructions=(
                "You are a loan underwriting assistant at a bank. "
                "Use your available skills to assess credit risk for loan applications. "
                "Always provide clear explanations of risk factors and recommendations."
            ),
            context_providers=[file_skills_provider],
        )

        print("=== 🏦 Credit Risk Assessment Agent ===")
        print("="*50)

        # Loan application scenario
        query = (
            "Assess the credit risk for this loan application:\n"
            "- Applicant credit score: 720\n"
            "- Annual income: $85,000\n"
            "- Monthly debt payments: $2,100\n"
            "- Requested loan amount: $250,000\n"
            "- Employment: 6 years at current employer\n"
            "- Collateral: Property valued at $300,000\n"
            "\nPlease calculate the risk score and provide a recommendation."
        )

        print(f"\n📋 Loan Application:\n{query}")
        print("-"*50)

        response = await agent.run(query)
        print(f"\n🤖 Agent Assessment:\n{response.text}")

await demo_file_based_skills()

## 3. Code-Defined Skills — Transaction Compliance

Code-defined skills are created inline using `Skill()`. They're useful when:
- Content is generated dynamically
- You want skills alongside application code
- Resources need to execute logic at read time

We'll create a compliance skill with:
- Dynamic resources (current transaction limits)
- Code-defined scripts (transaction screening)

In [ ]:
# Create a code-defined compliance skill
transaction_compliance_skill = InlineSkill(
    frontmatter=SkillFrontmatter(
        name="transaction-compliance",
        description=(
            "Screen financial transactions for AML/KYC compliance. "
            "Use when checking if transactions meet regulatory requirements."
        ),
    ),
    instructions=dedent("""\
        You are a compliance screening system. When asked to check a transaction:
        1. Run the `screen_transaction` script with the transaction details
        2. Check the `current_thresholds` resource for up-to-date limits
        3. Provide a clear compliance determination with reasoning
        4. Flag any items requiring Suspicious Activity Report (SAR) filing
    """),
)

# Dynamic resource — returns current compliance thresholds
@transaction_compliance_skill.resource(
    name="current-thresholds",
    description="Current regulatory reporting thresholds and limits",
)
def get_current_thresholds() -> str:
    """Return current compliance thresholds (could query a database in production)."""
    thresholds = {
        "ctr_threshold": 10000,
        "international_wire_reporting": 3000,
        "structuring_window_hours": 24,
        "structuring_aggregate_limit": 10000,
        "high_risk_countries": ["IR", "KP", "SY", "MM"],
        "pep_enhanced_due_diligence": True,
        "last_updated": "2025-01-15",
    }
    return json.dumps(thresholds, indent=2)

# Code-defined script — screens a transaction
@transaction_compliance_skill.script(
    name="screen_transaction",
    description="Screen a transaction for compliance. Args: amount (float), type (cash|wire|ach), country_code (str), is_pep (bool)",
)
def screen_transaction(
    amount: float,
    type: str = "cash",
    country_code: str = "US",
    is_pep: bool = False,
) -> str:
    """Screen a financial transaction against compliance rules."""
    flags = []
    risk_level = "LOW"

    # CTR threshold check
    if type == "cash" and amount > 10000:
        flags.append("CTR_REQUIRED: Cash transaction exceeds $10,000")
        risk_level = "MEDIUM"

    # Structuring detection (just below threshold)
    if type == "cash" and 8000 <= amount <= 10000:
        flags.append("STRUCTURING_ALERT: Transaction near CTR threshold")
        risk_level = "HIGH"

    # International wire reporting
    if type == "wire" and country_code != "US" and amount > 3000:
        flags.append(f"WIRE_REPORTING: International wire >$3,000 to {country_code}")
        if risk_level == "LOW":
            risk_level = "MEDIUM"

    # High-risk country
    high_risk = ["IR", "KP", "SY", "MM"]
    if country_code in high_risk:
        flags.append(f"HIGH_RISK_COUNTRY: {country_code} is on restricted list")
        risk_level = "CRITICAL"

    # PEP check
    if is_pep:
        flags.append("PEP_FLAG: Enhanced due diligence required")
        if risk_level in ("LOW", "MEDIUM"):
            risk_level = "HIGH"

    result = {
        "transaction": {"amount": amount, "type": type, "country": country_code, "pep": is_pep},
        "risk_level": risk_level,
        "flags": flags if flags else ["No compliance flags"],
        "action": "BLOCK" if risk_level == "CRITICAL" else "FLAG" if risk_level == "HIGH" else "PROCEED",
    }
    return json.dumps(result, indent=2)

print("✅ Code-defined transaction compliance skill created")
print(f"   Resources: current-thresholds (dynamic)")
print(f"   Scripts: screen_transaction")

In [ ]:
# Create a SkillsProvider with the code-defined skill
code_skills_provider = SkillsProvider([transaction_compliance_skill])

async def demo_code_defined_skills():
    """Demonstrate code-defined skills with transaction compliance."""
    async with AzureCliCredential(process_timeout=30) as credential:
        client = FoundryChatClient(credential=credential)
        agent = client.as_agent(
            name="ComplianceScreeningAgent",
            instructions=(
                "You are a compliance screening agent at a financial institution. "
                "Screen transactions for regulatory compliance using your available skills. "
                "Provide clear determinations and flag any suspicious activity."
            ),
            context_providers=[code_skills_provider],
        )

        print("=== 🛡️ Transaction Compliance Screening Agent ===")
        print("="*50)

        # Test transactions
        transactions = [
            "Screen this transaction: $15,000 cash deposit from a domestic customer.",
            "Check compliance for: $50,000 wire transfer to Iran (country code IR).",
            "Is this suspicious? Three cash deposits of $9,500 each within 24 hours from the same account.",
        ]

        for txn in transactions:
            print(f"\n📋 Transaction: {txn}")
            print("-"*50)
            response = await agent.run(txn)
            print(f"🤖 Compliance Result: {response.text}")

await demo_code_defined_skills()

## 4. Combined Skills — Full FSI Agent

Combine file-based and code-defined skills in a single `SkillsProvider` for a comprehensive FSI agent that can:
- Assess credit risk (file-based skill)
- Screen transactions for compliance (code-defined skill)
- Check regulatory requirements (file-based skill with references)

In [ ]:
# Combined provider: file-based FSI skills + code-defined compliance skill
combined_skills_provider = SkillsProvider(
    DeduplicatingSkillsSource(
        AggregatingSkillsSource([
            FileSkillsSource(
                Path(".").absolute() / "fsi-skills",
                script_runner=fsi_script_runner,
            ),
            InMemorySkillsSource([transaction_compliance_skill]),
        ])
    ),
)

async def demo_combined_skills():
    """Demonstrate combined file-based and code-defined skills."""
    async with AzureCliCredential(process_timeout=30) as credential:
        client = FoundryChatClient(credential=credential)
        agent = client.as_agent(
            name="FSIAdvisorAgent",
            instructions=(
                "You are a comprehensive financial services AI assistant. "
                "You can assess credit risk, screen transactions for compliance, "
                "and provide regulatory guidance. Use your available skills as needed. "
                "Always note that your assessments are for informational purposes only."
            ),
            context_providers=[combined_skills_provider],
        )

        print("=== 🏦 Combined FSI Agent (Credit Risk + Compliance) ===")
        print("="*60)

        # Complex scenario combining multiple skills
        scenario = (
            "A customer is applying for a $500,000 commercial loan. "
            "Their credit score is 680, annual income is $150,000, "
            "monthly debt is $4,500, employed for 3 years, "
            "and offering a property worth $400,000 as collateral. "
            "Additionally, they recently made a $12,000 wire transfer to Germany. "
            "Please assess their loan application and flag any compliance concerns."
        )

        print(f"\n📋 Scenario: {scenario}")
        print("-"*60)

        response = await agent.run(scenario)
        print(f"\n🤖 FSI Agent Assessment:\n{response.text}")

await demo_combined_skills()

## Summary

| Concept | Details |
|---------|----------|
| **File-based Skills** | Discovered from `SKILL.md` in filesystem directories |
| **Code-defined Skills** | Created inline with `Skill()`, `@skill.resource`, `@skill.script` |
| **SkillsProvider** | Context provider that manages skill lifecycle |
| **Progressive Disclosure** | Advertise → Load → Read → Run (minimizes context usage) |
| **Script Runner** | Custom callable for executing file-based scripts |
| **Combined Skills** | Mix file-based and code-defined in one provider |

### Key Patterns

```python
# File-based: discover from directory
provider = SkillsProvider(skill_paths=Path("skills/"), script_runner=runner)

# Code-defined: inline with decorators
skill = Skill(name="my-skill", description="...", content="...")

@skill.resource
def data() -> str: return "dynamic content"

@skill.script(name="run", description="...")
def execute(arg: str) -> str: return f"result: {arg}"

# Combined
provider = SkillsProvider(skill_paths=Path("skills/"), skills=[skill], script_runner=runner)
```

### Next Steps
- **Hosted Agent with Skills**: See `../../azure-ai-agents/10-hosted-agent-with-skills.ipynb`
- **Workflows**: See `../workflows/` for multi-agent orchestration
- **Agent Skills Specification**: [agentskills.io](https://agentskills.io/)

## 📚 References

- [Agent skills](https://learn.microsoft.com/agent-framework/agents/skills) — File-based and code-defined SKILL.md skills
- [Adding skills](https://learn.microsoft.com/agent-framework/journey/adding-skills) — Walkthrough of attaching skills to an agent
- [Skills vs workflows](https://learn.microsoft.com/agent-framework/agents/skills#when-to-use-skills-vs-workflows) — Choosing skills vs workflows; progressive disclosure
- [Agent Framework overview](https://learn.microsoft.com/agent-framework/overview/) — Framework concepts and capabilities

> 💡 Tip: search the official docs live via the **Microsoft Learn MCP** at `https://learn.microsoft.com/api/mcp`.